# Week 6 — Advanced Retrieval: Transform and Measure
Self-contained extension of the Week 5 retriever using the same 16 chunks and frozen 10-query evaluation set. The notebook measures baseline retrieval, Gemini multi-query transformation, transformation-only retrieval, cross-encoder reranking, per-query deltas, query-type effects, and a transformation failure case.

**Submission:** no sidecar cache file is required. Rewrites are cached in an in-notebook dictionary during execution and printed into notebook output. Export the executed notebook to HTML for Canvas.


In [ ]:
import sys,subprocess,importlib.util,os,json,time,math,re
mods={"numpy":"numpy","pandas":"pandas","sentence_transformers":"sentence-transformers","faiss":"faiss-cpu","google.genai":"google-genai","dotenv":"python-dotenv"}
need=[p for m,p in mods.items() if importlib.util.find_spec(m) is None]
if need: subprocess.check_call([sys.executable,"-m","pip","install",*need])
import numpy as np,pandas as pd,faiss
from collections import defaultdict
from dotenv import load_dotenv,find_dotenv
from sentence_transformers import SentenceTransformer,CrossEncoder
from IPython.display import display,Markdown
TOP_K=3; CANDIDATE_K=8
EMBED_MODEL="sentence-transformers/all-MiniLM-L6-v2"
RERANK_MODEL="cross-encoder/ms-marco-MiniLM-L-6-v2"
GEMINI_MODEL="gemini-3.5-flash-lite"


## 1. Week 5 corpus and frozen evaluation set

In [ ]:
RAW=[
("D01-S1","Request intake — Required fields. A new operations request must include a short title, requester name, business team, problem statement, and desired outcome. The requester can attach example files but attachments are optional."),
("D01-S2","Request intake — Duplicate requests. Before opening a new request, search the request list by title and team. If an existing request covers the same problem, add a comment to that request rather than creating a duplicate."),
("D02-S1","Triage and priority — Priority definitions. Priority 1 means a production service is unavailable to multiple people. Priority 2 means a blocked business process with a manual workaround. Priority 3 means an improvement or a question without an active outage."),
("D02-S2","Triage and priority — First review. The operations coordinator checks completeness and assigns an owner within two business days. Missing intake fields are sent back to the requester before priority is assigned."),
("D03-S1","Approval workflow — Approval conditions. A request that changes customer-facing content or grants access to protected data requires manager approval before implementation. An internal wording correction with no data access change does not require approval."),
("D03-S2","Approval workflow — Rejected approval. When a manager rejects a request, record the reason and set approval result to Rejected. The owner may revise the request and seek a new approval; rejection does not mean it is complete."),
("D04-S1","Delivery and review — Human review. Generated drafts must be reviewed for factual correctness by the assigned owner. A workflow must not automatically send a draft to customers or publish it internally before a person approves the final text."),
("D04-S2","Delivery and review — Completion. Set status to Completed only after the requester accepts the delivered result and the owner records the outcome. Saving a generated draft alone does not close a request."),
("D05-S1","SharePoint file handling — Source files. Store source files in the team SharePoint document library. In the request, link the specific file and record its version; do not paste the full file into the request description."),
("D05-S2","SharePoint file handling — Access permissions. Requesters may see their own request and shared team files. An owner checks a recipient's access before sharing a source link; a link does not grant permission by itself."),
("D06-S1","Automation monitoring — Flow failure. If a Power Automate run fails, inspect its run history and note the failed action and error in the request. Retry only after correcting the cause, to avoid sending duplicate notifications."),
("D06-S2","Automation monitoring — Success checks. For a new automation, test one ordinary case and one missing-field case. Record expected versus observed results and ask the owner to sign off before activation."),
("D07-S1","Data safeguards — Sensitive content. Remove customer identifiers and confidential attachments from example prompts unless their use is authorized. Use a small, sanitized excerpt when a full document is unnecessary."),
("D07-S2","Data safeguards — Retention. Keep the request's decision and final outcome in the request list. Delete temporary test exports when validation finishes according to the team's retention schedule."),
("D08-S1","Notifications and escalation — Status updates. The owner posts a status update on the request when work starts and when the expected delivery date changes. Routine updates are visible to the requester in the list."),
("D08-S2","Notifications and escalation — Escalation. Escalate a production outage to the on-call owner immediately and record the escalation time. A non-outage item waiting more than two business days for assignment goes to the operations coordinator.")
]
CHUNKS=[{"chunk_id":i,"text":t} for i,t in RAW]; by_id={c["chunk_id"]:c for c in CHUNKS}
TESTS=[
("Q01","What fields must a requester provide when opening a request?",{"D01-S1"}),
("Q02","What should I do if someone has already reported the same problem?",{"D01-S2"}),
("Q03","Which priority applies when a process is blocked but a workaround exists?",{"D02-S1"}),
("Q04","How soon is an owner assigned, and what happens to an incomplete request?",{"D02-S2"}),
("Q05","When does a request require manager approval?",{"D03-S1"}),
("Q06","What happens after a manager rejects a request?",{"D03-S2"}),
("Q07","Can an AI draft be published or emailed without a person checking it?",{"D04-S1"}),
("Q08","When can the owner mark the request Completed?",{"D04-S2"}),
("Q09","How do I investigate a failed Power Automate flow before retrying?",{"D06-S1"}),
("Q10","How quickly should a production outage be escalated?",{"D08-S2"})]
display(pd.DataFrame([{"qid":q,"query":x,"gold":next(iter(g))} for q,x,g in TESTS]))


## 2. Week 5 baseline

In [ ]:
embedder=SentenceTransformer(EMBED_MODEL)
V=embedder.encode([c["text"] for c in CHUNKS],normalize_embeddings=True,show_progress_bar=False).astype("float32")
index=faiss.IndexFlatIP(V.shape[1]); index.add(V)
def retrieve(q,k=TOP_K):
    v=embedder.encode([q],normalize_embeddings=True,show_progress_bar=False).astype("float32")
    s,idx=index.search(v,k)
    return [{"chunk_id":CHUNKS[j]["chunk_id"],"score":float(sc),"rank":r+1} for r,(j,sc) in enumerate(zip(idx[0],s[0]))]
baseline={qid:retrieve(q) for qid,q,_ in TESTS}
display(pd.DataFrame([{"qid":qid,"baseline":[x["chunk_id"] for x in baseline[qid]]} for qid,_,_ in TESTS]))


## 3. Gemini multi-query transformation
Gemini produces two unique rewrites per query. The original query stays in reciprocal-rank fusion. Rewrites are cached only inside this notebook execution and printed in full so the executed notebook/HTML preserves them.


In [ ]:
def key():
    try:
        from google.colab import userdata
        for n in ("GEMINI_API_KEY","GEMENI_API_KEY"):
            try:
                v=userdata.get(n)
                if v:return v.strip(),f"Colab Secrets ({n})"
            except:pass
    except ImportError:pass
    for n in ("GEMINI_API_KEY","GEMENI_API_KEY"):
        v=os.getenv(n)
        if v:return v.strip(),f"environment ({n})"
    f=find_dotenv(usecwd=True)
    if f:
        load_dotenv(f,override=False)
        for n in ("GEMINI_API_KEY","GEMENI_API_KEY"):
            v=os.getenv(n)
            if v:return v.strip(),f".env ({n})"
    return None,None
def uniq(vals):
    out=[]
    for v in vals or []:
        if isinstance(v,str):
            v=re.sub(r"^\s*\d+[.):-]\s*","",v).strip().strip('"')
            if v and v not in out:out.append(v)
    return out
api,src=key()
if not api:raise RuntimeError("Add GEMINI_API_KEY to Colab Secrets, the environment, or local .env.")
print("Gemini API key loaded from",src)
from google import genai
client=genai.Client(api_key=api)
def rewrites(q):
    p=f"Rewrite this semantic-search query two different ways using likely operations-policy vocabulary. Preserve intent exactly. Return exactly two concise queries, one per line, prefixed 1. and 2.\n\nOriginal query: {q}"
    for _ in range(4):
        try:
            r=uniq(client.models.generate_content(model=GEMINI_MODEL,contents=p).text.splitlines())
            if len(r)>=2:return r[:2]
            raise ValueError("Fewer than two unique rewrites")
        except Exception as e:
            m=str(e)
            if "429" in m or "RESOURCE_EXHAUSTED" in m:
                x=re.search(r"retry in ([0-9.]+)s",m,re.I); d=float(x.group(1))+2 if x else 40
                print(f"Rate limit; retrying in {d:.0f}s"); time.sleep(d); continue
            raise
    raise RuntimeError("Gemini did not return two unique rewrites.")
query_cache={}
for qid,q,_ in TESTS:
    if len(uniq(query_cache.get(qid)))<2:
        print("Generating",qid); query_cache[qid]=rewrites(q); time.sleep(2)
transformed={qid:uniq(query_cache[qid])[:2] for qid,_,_ in TESTS}
assert all(len(v)==2 and len(set(v))==2 for v in transformed.values())
print("In-notebook query rewrite cache:\n"+json.dumps(transformed,indent=2))
display(pd.DataFrame([{"qid":qid,"original":q,"rewrite_1":transformed[qid][0],"rewrite_2":transformed[qid][1]} for qid,q,_ in TESTS]))


## 4. Transformation-only retrieval and cross-encoder reranking

In [ ]:
def rrf(lists,k=60):
    d=defaultdict(float)
    for ranked in lists:
        for x in ranked:d[x["chunk_id"]]+=1/(k+x["rank"])
    return [cid for cid,_ in sorted(d.items(),key=lambda z:z[1],reverse=True)]
cands={qid:rrf([retrieve(q,CANDIDATE_K)]+[retrieve(r,CANDIDATE_K) for r in transformed[qid]])[:CANDIDATE_K] for qid,q,_ in TESTS}
transform_only={qid:v[:TOP_K] for qid,v in cands.items()}
reranker=CrossEncoder(RERANK_MODEL)
def rerank(q,ids):
    s=reranker.predict([(q,by_id[i]["text"]) for i in ids])
    return [i for i,_ in sorted(zip(ids,s),key=lambda z:float(z[1]),reverse=True)[:TOP_K]]
finals={qid:rerank(q,cands[qid]) for qid,q,_ in TESTS}
display(pd.DataFrame([{"qid":qid,"transform_only":transform_only[qid],"final":finals[qid]} for qid,_,_ in TESTS]))


## 5. Before/after metrics, query types, and required failure case

In [ ]:
def met(ids,gold):
    h=[int(x in gold) for x in ids[:TOP_K]]
    rr=next((1/i for i,x in enumerate(ids[:TOP_K],1) if x in gold),0)
    dcg=sum(r/math.log2(i+2) for i,r in enumerate(h))
    return {"p3":sum(h)/TOP_K,"r3":sum(h)/len(gold),"hit1":h[0],"mrr":rr,"ndcg3":dcg}
rows=[]
for qid,q,g in TESTS:
    b=[x["chunk_id"] for x in baseline[qid]]; t=transform_only[qid]; f=finals[qid]
    bm,tm,fm=met(b,g),met(t,g),met(f,g)
    r={"qid":qid,"query":q,"gold":next(iter(g)),"baseline_ids":b,"transform_ids":t,"final_ids":f}
    for pre,m in (("base",bm),("transform",tm),("final",fm)):
        r.update({f"{pre}_{k}":v for k,v in m.items()})
    r["delta_transform_mrr"]=tm["mrr"]-bm["mrr"]; r["delta_final_mrr"]=fm["mrr"]-bm["mrr"]; rows.append(r)
results=pd.DataFrame(rows)
display(results[["qid","query","gold","baseline_ids","transform_ids","final_ids","base_hit1","transform_hit1","final_hit1","delta_transform_mrr","delta_final_mrr"]])
names=["p3","r3","hit1","mrr","ndcg3"]
summary=pd.DataFrame({"baseline":[results[f"base_{m}"].mean() for m in names],"transform_only":[results[f"transform_{m}"].mean() for m in names],"final_reranked":[results[f"final_{m}"].mean() for m in names]},index=names)
summary["transform_delta"]=summary.transform_only-summary.baseline; summary["final_delta"]=summary.final_reranked-summary.baseline
display(summary.round(3))
types=pd.DataFrame([{"qid":r.qid,"query_type":"baseline gold not rank 1" if r.base_hit1==0 else "baseline already rank 1","transform_mrr_delta":r.delta_transform_mrr,"final_mrr_delta":r.delta_final_mrr} for _,r in results.iterrows()])
display(types.groupby("query_type",observed=True).agg(queries=("qid","count"),mean_transform_mrr_delta=("transform_mrr_delta","mean"),mean_final_mrr_delta=("final_mrr_delta","mean")).round(3))
reg=results[results.delta_transform_mrr<0].sort_values("delta_transform_mrr")
if len(reg):
    w=reg.iloc[0]; qid=w.qid
    display(Markdown(f"### Transformation regression: {qid}\n**Original query:** {w['query']}\n\n**Rewrites:** {transformed[qid]}\n\n**Gold chunk:** {w['gold']} — {by_id[w['gold']]['text']}\n\n**Baseline ranking:** {w['baseline_ids']}\n\n**Transformation-only ranking:** {w['transform_ids']}\n\n**Final reranked ranking:** {w['final_ids']}\n\n**Explanation:** The rewrite appears to have emphasized a related operations concept enough to move a less-direct chunk above the gold chunk. This is an inference from the observed ranking change, not proof of the embedding model's internal cause."))
else:
    display(Markdown("### No transformation regression observed\nNo query had lower MRR after transformation-only RRF than baseline. The original query remained in fusion and the rewrites preserved the requested action/entity across this set."))


## 6. Measured interpretation

In [ ]:
b=summary.baseline;t=summary.transform_only;f=summary.final_reranked
display(Markdown(f"- **Baseline:** P@3 {b.p3:.3f}, R@3 {b.r3:.3f}, Hit@1 {b.hit1:.3f}, MRR {b.mrr:.3f}, nDCG@3 {b.ndcg3:.3f}.\n- **Transformation-only:** P@3 {t.p3:.3f}, R@3 {t.r3:.3f}, Hit@1 {t.hit1:.3f}, MRR {t.mrr:.3f}, nDCG@3 {t.ndcg3:.3f}.\n- **Final reranked:** P@3 {f.p3:.3f}, R@3 {f.r3:.3f}, Hit@1 {f.hit1:.3f}, MRR {f.mrr:.3f}, nDCG@3 {f.ndcg3:.3f}.\n\nTransformation MRR delta: **{summary.loc['mrr','transform_delta']:+.3f}**. Final MRR delta: **{summary.loc['mrr','final_delta']:+.3f}**.\n\nThe cross-encoder adds CPU latency because it jointly scores query-candidate pairs; that cost is justified when several plausible chunks need precise ordering."))


## AI use
ChatGPT was used to debug the Week 6 notebook, preserve the Week 5 evaluation set, and grammatically check/improve documentation. Gemini Flash-Lite is used for the measured query transformations. The analysis, measurements, and final decisions are based on the executed notebook results.
